# 05 — Seven-Day Capacity Forecasting Baseline

Train a dependency-light ridge regression benchmark with a chronological split
and seven-row gap. Compare it with a naive “current load” forecast.

In [ ]:
from pathlib import Path
import sys
import numpy as np  # noqa: F401 -- shared setup; used by modeling notebooks
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR = PROJECT_ROOT / "output"
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

In [ ]:
import json
from app_utils import TOTAL_LOAD_COLUMN

feature_data = pd.read_parquet(PROCESSED_DIR / "uac_capacity_ml_features.parquet").sort_index()
target_column = "target_total_load_t_plus_7d"
prefixes = ("calendar_", "operational_", "lag_", "rolling_", "ema_", "momentum_", "quality_")
feature_columns = [
    column
    for column in feature_data.columns
    if column.startswith(prefixes) and pd.api.types.is_numeric_dtype(feature_data[column])
]
working = (
    feature_data[feature_columns + [target_column, TOTAL_LOAD_COLUMN]]
    .replace([np.inf, -np.inf], np.nan)
    .dropna(subset=[target_column])
)

test_start = int(np.floor(len(working) * 0.80))
train_end = test_start - 7
train = working.iloc[:train_end]
test = working.iloc[test_start:]
X_train = train[feature_columns].to_numpy(dtype=float)
X_test = test[feature_columns].to_numpy(dtype=float)
y_train = train[target_column].to_numpy(dtype=float)
y_test = test[target_column].to_numpy(dtype=float)

medians = np.nanmedian(X_train, axis=0)
usable = np.isfinite(medians)
feature_columns = [name for name, keep in zip(feature_columns, usable, strict=True) if keep]
medians = medians[usable]
X_train = X_train[:, usable]
X_test = X_test[:, usable]
X_train = np.where(np.isnan(X_train), medians, X_train)
X_test = np.where(np.isnan(X_test), medians, X_test)

means = X_train.mean(axis=0)
scales = X_train.std(axis=0)
scales = np.where(scales > 1e-12, scales, 1.0)
X_train_scaled = (X_train - means) / scales
X_test_scaled = (X_test - means) / scales

alpha = 10.0
y_mean = y_train.mean()
identity = np.eye(X_train_scaled.shape[1])
coefficients = np.linalg.solve(
    X_train_scaled.T @ X_train_scaled + alpha * identity,
    X_train_scaled.T @ (y_train - y_mean),
)
predictions = X_test_scaled @ coefficients + y_mean
naive_predictions = test[TOTAL_LOAD_COLUMN].to_numpy(dtype=float)

In [ ]:
def regression_metrics(actual, predicted):
    residual = actual - predicted
    denominator = np.where(np.abs(actual) > 1e-12, np.abs(actual), np.nan)
    return {
        "mae": float(np.mean(np.abs(residual))),
        "rmse": float(np.sqrt(np.mean(residual**2))),
        "mape_percent": float(np.nanmean(np.abs(residual) / denominator) * 100),
        "r2": float(1 - np.sum(residual**2) / np.sum((actual - actual.mean()) ** 2)),
    }


ridge_metrics = regression_metrics(y_test, predictions)
naive_metrics = regression_metrics(y_test, naive_predictions)
evaluation = {
    "model": "NumPy ridge regression baseline",
    "target": target_column,
    "forecast_horizon_days": 7,
    "alpha": alpha,
    "training_rows": len(train),
    "test_rows": len(test),
    "gap_rows": 7,
    "feature_count": len(feature_columns),
    "training_start": train.index.min().date().isoformat(),
    "training_end": train.index.max().date().isoformat(),
    "test_start": test.index.min().date().isoformat(),
    "test_end": test.index.max().date().isoformat(),
    "ridge": ridge_metrics,
    "naive_current_load": naive_metrics,
    "mae_improvement_percent": float(
        (naive_metrics["mae"] - ridge_metrics["mae"]) / naive_metrics["mae"] * 100
    ),
    "champion_model": "ridge"
    if ridge_metrics["mae"] < naive_metrics["mae"]
    else "naive_current_load",
    "ridge_deployment_recommendation": "continue_research"
    if ridge_metrics["mae"] < naive_metrics["mae"]
    else "do_not_promote",
}
evaluation

In [ ]:
model_dir = OUTPUT_DIR / "models"
export_dir = OUTPUT_DIR / "exports"
model_dir.mkdir(parents=True, exist_ok=True)
export_dir.mkdir(parents=True, exist_ok=True)

np.savez_compressed(
    model_dir / "capacity_ridge_baseline.npz",
    coefficients=coefficients,
    intercept=np.array([y_mean]),
    feature_names=np.asarray(feature_columns, dtype=str),
    medians=medians,
    means=means,
    scales=scales,
    alpha=np.array([alpha]),
    forecast_horizon_days=np.array([7]),
)
(model_dir / "evaluation_metrics.json").write_text(
    json.dumps(evaluation, indent=2, allow_nan=False) + "\n",
    encoding="utf-8",
)
prediction_frame = pd.DataFrame(
    {
        "Date": test.index,
        "Actual Total System Load T+7": y_test,
        "Ridge Prediction": predictions,
        "Naive Current Load Prediction": naive_predictions,
        "Ridge Residual": y_test - predictions,
    }
)
prediction_frame.to_csv(
    export_dir / "model_test_predictions.csv",
    index=False,
    date_format="%Y-%m-%d",
)
print(f"Saved model and {len(prediction_frame)} chronological test predictions.")

This baseline is for research comparison. Before deployment,
add rolling-origin evaluation, uncertainty intervals, drift monitoring, and an
explicit approval process for any operational use.